# OOI helpers: availability, fetch, plot, analyze

Run this notebook from the project folder so `import ooi_tools` resolves. Get the exact `node / sensor / method / stream` from the `ooi_lookup` tool in the chat.

Two data sources: `"thredds"` (public Gold Copy server, works anywhere) and `"kdata"` (NetCDF files mounted on an OOI JupyterHub session). The main flow below uses THREDDS; the last section shows the same steps with kdata.

In [ ]:
from ooi_tools import ooi_availability, ooi_fetch, ooi_plot, ooi_series

# 1) Date coverage of the stream (no download)
ooi_availability("CE02SHSM", "RID27", "02-FLORTD000", "telemetered", "flort_sample")

In [ ]:
# 2) Fetch a period inside that range from THREDDS (the default source)
info = ooi_fetch("CE02SHSM", "RID27", "02-FLORTD000", "telemetered", "flort_sample",
                 start="2024-01-01", stop="2024-06-30", source="thredds")
print(info["status"])
info.get("science_variables")   # science variables labeled; engineering columns hidden

In [ ]:
# 3) Plot a science variable (pass several site codes to overlay a comparison)
ooi_plot("fluorometric_chlorophyll_a", sites=["CE02SHSM"], title="Oregon Shelf chlorophyll")

In [ ]:
# 4) Analyze: ooi_series returns a time-indexed pandas Series.
#    Seasonal cycle of chlorophyll (a bloom is the seasonal maximum).
s = ooi_series("fluorometric_chlorophyll_a")
print("peak month:", s.resample("1M").mean().idxmax())
s.groupby(s.index.month).mean().round(2)

## Same steps from kdata (OOI JupyterHub)

On the OOI JupyterHub the NetCDF files are mounted at `~/ooi/kdata`, so `source="kdata"` reads them locally instead of downloading from THREDDS. The calls are identical apart from `source`. Off the Hub these cells return `not_found` because the mount is not present.

In [ ]:
# Date coverage from the kdata mount (no download)
ooi_availability("CE02SHSM", "RID27", "02-FLORTD000", "telemetered", "flort_sample",
                 source="kdata")

In [ ]:
# Fetch from kdata, then plot/analyze exactly as above
info = ooi_fetch("CE02SHSM", "RID27", "02-FLORTD000", "telemetered", "flort_sample",
                 start="2024-01-01", stop="2024-06-30", source="kdata")
print(info["status"])
info.get("science_variables")